# FAISS (Facebook AI Similarity Search)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from plantclef.config import get_device

print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Using device: {device}")

PyTorch Version: 2.6.0+cu124
Using device: cuda


In [3]:
import pandas as pd
from pathlib import Path

# Get list of stored filed in cloud bucket
root = Path().resolve().parents[0]
print(root)
! date

/storage/home/hcoda1/9/mgustineli3/clef/pytorch-plantclef
Mon Mar 24 10:41:56 AM EDT 2025


In [4]:
# path to data
data_path = f"{root}/data/embeddings"
train_path = f"{data_path}/train_embeddings"
test_path = f"{data_path}/test_grid_3x3_embeddings"

# read train/test data
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

# display data
print(f"Train DF shape: {train_df.shape}")
print(f"Test DF shape: {test_df.shape}")
display(train_df.head(3))
display(test_df.head(3))

Train DF shape: (2020, 8)
Test DF shape: (3600, 6)


,image_name,data,species,species_id,embeddings,logits,tile,partition
0,28e2fbd0cc93d82d7de3ed5783c64816074955e0.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Pteridium aquilinum (L.) Kuhn,1356404,"[0.22075553238391876, -0.5152621269226074, 0.7...","{""1389294"": 0.4966914653778076, ""1356390"": 0.0...",0,0
1,47b5db375e741fdc09b5e133d5bfef0285aa695c.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Carex firma Host,1418612,"[0.55893474817276, 0.19274836778640747, 0.8937...","{""1418612"": 0.7230438590049744, ""1390910"": 0.0...",0,0
2,c6a6b172ab03374ad1bb713b7e7b37dd84f094a1.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Daucus pumilus (L.) Hoffmanns. & Link,1722578,"[-0.15363329648971558, 0.35160958766937256, 0....","{""1722578"": 0.8078303337097168, ""1411700"": 0.0...",0,0


,image_name,data,embeddings,logits,tile,partition
0,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-1.3391107320785522, 1.6340396404266357, -2.2...","{""1741880"": 0.1965922713279724, ""1729043"": 0.0...",0,0
1,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-0.11381672322750092, 1.790043830871582, -0.5...","{""1395807"": 0.19840383529663086, ""1741880"": 0....",1,0
2,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-0.40855133533477783, 1.9910684823989868, -0....","{""1395807"": 0.4447108805179596, ""1397468"": 0.0...",2,0


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   image_name  2020 non-null   object  
 1   data        2020 non-null   object  
 2   species     2020 non-null   object  
 3   species_id  2020 non-null   int32   
 4   embeddings  2020 non-null   object  
 5   logits      2020 non-null   object  
 6   tile        2020 non-null   int64   
 7   partition   2020 non-null   category
dtypes: category(1), int32(1), int64(1), object(5)
memory usage: 105.2+ KB


In [37]:
import faiss
import torch

idx2cls = train_df["species_id"].values
# convert embeddings to tensor
embs = torch.tensor(train_df["embeddings"].tolist()).float().to(device)
# normalize embeddings for cosine similarity
embs = torch.nn.functional.normalize(embs, p=2, dim=1)
# create FAISS index
index = faiss.IndexFlatIP(embs.shape[1])  # inner product (dot product)
index.add(embs.cpu().numpy())  # FAISS expects numpy arrays

In [38]:
(
    embs.shape,
    idx2cls.shape,
)

(torch.Size([2020, 768]), (2020,))

In [39]:
# normalize embeddings for cosine similarity
query_embs = torch.nn.functional.normalize(embs, p=2, dim=1)
# perform search
similarities, indices = index.search(query_embs.cpu().numpy(), k=1)
predictions = idx2cls[indices.flatten()]
predictions.shape

(2020,)

In [40]:
predictions

array([1356404, 1418612, 1722578, ..., 1388766, 1397500, 1393687],
      dtype=int32)

In [41]:
import torch
import faiss
from plantclef.config import get_device


class FaissClassifier:
    def __init__(self, train_df: pd.DataFrame):
        """
        :param train_df: DataFrame with columns ["species_id", "embeddings"]
        """
        self.device = get_device()
        self.index, self.idx2cls = self.build_index(train_df)

    def build_index(self, train_df):
        """Builds the FAISS index from the training data."""

        idx2cls = train_df["species_id"].values
        # convert embeddings to tensor
        embs = torch.tensor(
            train_df["embeddings"].tolist(), dtype=torch.float32, device=self.device
        )
        # normalize embeddings for cosine similarity
        embs = torch.nn.functional.normalize(embs, p=2, dim=1)
        # create FAISS index
        index = faiss.IndexFlatIP(embs.shape[1])  # inner product (dot product)
        index.add(embs.cpu().numpy())  # FAISS expects numpy arrays
        return index, idx2cls

    def make_prediction(self, query_embeddings: torch.Tensor, k=1):
        """
        Predicts the class of given embeddings using nearest neighbor search.
        :param query_embeddings: tensor of shape (N, D) where N is the number of embeddings and D is the embedding dimension
        :param k: number of nearest neighbors to return
        :return: predictions, similarities
        """

        # normalize embeddings for cosine similarity
        query_embeddings = torch.nn.functional.normalize(query_embeddings, p=2, dim=1)
        # perform search
        similarities, indices = self.index.search(query_embeddings.cpu().numpy(), k=k)
        predictions = idx2cls[indices.flatten()]
        return predictions, similarities

In [42]:
# similarity seearch using FAISS
nn_classifier = FaissClassifier(train_df)
query_embs = torch.tensor(
    test_df["embeddings"].tolist(), dtype=torch.float32, device=get_device()
)
cls, conf = nn_classifier.make_prediction(query_embs, k=1)

In [43]:
cls.shape, conf.shape

((3600,), (3600, 1))